# AI Engagement Sortition Selection

This notebook runs the sortition algorithm to select a representative panel from survey respondents. It:
1. Fetches stratification targets and candidate data from Snowflake
2. Runs the sortition algorithm with configured constraints
3. Writes the selected panel back to Snowflake

## Configuration

Set algorithm parameters, deviation tolerances, and the target Snowflake table for panel output. 

In [ ]:
# Sortition settings
selection_algorithm = "maximin"  # default is maximin
id_column = "SURVEY_RESPONDENT_ID"
columns_to_keep = []  # additional columns to keep in the output
number_people_wanted = 150
allowed_deviation = 0.25  # how much deviation from the target distribution we allow
additional_fow_tolerance = 0.25  # how much additional tolerance is given for the field_of_work category

# Groups that have already been filled in previous sortition rounds and should be entirely excluded in this sortition round.
groups_already_filled_list = []  # example: ['Government',]
groups_already_filled = ",".join(f"'{item}'" for item in groups_already_filled_list)

# Groups with a high need in this sortition round. The algorithm will try to invite all remaining unselected candidates from these groups.
groups_high_need = []   # example: ['18-24', 'Hospitality']

# Set a specific higher maximum number of candidates for some groups 
max_adjustment = 30
groups_to_adjust_max = []  # example: ['Education', 'Healthcare']

# Snowflake target configuration
TARGET_DATABASE = "TRANSFORM_ENGCA_DEV"  # "TRANSFORM_ENGCA_DEV" or "TRANSFORM_ENGCA_PRD"
TARGET_SCHEMA = "AI_ENGAGEMENT"
TARGET_TABLE = "INT_AI_ENGAGEMENT_SORTITION_SELECTIONS"

In [ ]:
%%sql -r targets_df
SELECT
    question as category,
    answer as name,
    adjusted_target_pct
FROM TRANSFORM_ENGCA_PRD.GOVOCAL.INT_GOVOCAL_SORTITION_TARGETS
where answer NOT IN ({{groups_already_filled}})

Next we want to get the people who are candidates for the sortition run. We only want to include candidates who have not already been selected, so we filter out any candidates from previous sortition runs. This also provides a way to exclude candidates from groups that are already over target, `groups_already_filled`.

In [ ]:
%%sql -r people_df
SELECT *
FROM TRANSFORM_ENGCA_PRD.GOVOCAL.INT_GOVOCAL_SORTITION_CANDIDATES
WHERE 
    survey_respondent_id NOT IN (
        SELECT DISTINCT survey_respondent_id FROM TRANSFORM_ENGCA_PRD.AI_ENGAGEMENT.INT_AI_ENGAGEMENT_SORTITION_SELECTIONS)
    and lower(invitee_status) <> 'accepted'
    and ARRAY_SIZE(
            ARRAY_INTERSECTION(
                ARRAY_CONSTRUCT(age, gender_category, race_ethnicity_category, region, field_of_work, ai_response_label), 
                ARRAY_CONSTRUCT({{groups_already_filled}})
                )
            ) = 0

## Demographic Distribution of Candidates and Acceptances

Count how many candidates and already-accepted participants fall into each demographic group. These counts feed into the tolerance-band calculation below.

In [ ]:
%%sql -r candidate_demo_count_df
SELECT
    question as category,
    answer as name,
    count(*) as candidate_count
FROM {{people_df}}
UNPIVOT (
    answer for question in (age, gender_category, race_ethnicity_category, region, field_of_work, ai_response_label)
)
GROUP BY question, answer

In [ ]:
%%sql -r accepted_demo_count_df
SELECT
    question as category,
    answer as name,
    count(*) as accepted_count
FROM TRANSFORM_ENGCA_PRD.GOVOCAL.INT_GOVOCAL_SORTITION_CANDIDATES
UNPIVOT (
    answer for question in (age, gender_category, race_ethnicity_category, region, field_of_work, ai_response_label)
)
WHERE 
    invitee_status = 'accepted'
GROUP BY question, answer

In [ ]:
%%sql -r total_accepted_count_df
SELECT
    count(*)
FROM TRANSFORM_ENGCA_PRD.GOVOCAL.INT_GOVOCAL_SORTITION_CANDIDATES
WHERE
    invitee_status = 'accepted'

## Calculate the min and max for each sortition group

We compute the remaining target for each demographic cell by finding the difference from the targeted acceptances and the actual acceptances for each category and group and adding this difference to the targeted number of candidates in this sortition batch. Then we define a tolerance band around that remaining need so the algorithm can select a feasible batch. There is also special handling for field of work groups if a larger tolerance is needed for this category, and special handling for groups that are high need or that need a specific maximum to be set.

In [ ]:
import pandas as pd

total_accepted_count = total_accepted_count_df.iloc[0, 0]

features_df = targets_df.merge(candidate_demo_count_df, on=["CATEGORY", "NAME"], how="left",)
features_df = features_df.merge(accepted_demo_count_df, on=["CATEGORY", "NAME"], how="left",)

features_df["CANDIDATE_COUNT"] = features_df["CANDIDATE_COUNT"].fillna(0).astype(int)
features_df["ACCEPTED_COUNT"] = features_df["ACCEPTED_COUNT"].fillna(0).astype(int)

# Calculate how far off of our target proportions we already are for those people who are currently accepted
features_df["PROPORTIONAL_ACCEPTED_COUNT"] = features_df["ADJUSTED_TARGET_PCT"] * total_accepted_count
features_df["DIFFERENCE_FROM_ACCEPTANCES"] = features_df["PROPORTIONAL_ACCEPTED_COUNT"] - features_df["ACCEPTED_COUNT"]

# Calculate what the targeted number of people for each category based on the number of people in this batch. And add the difference calculated above.
features_df["TARGET"] = features_df["ADJUSTED_TARGET_PCT"] * number_people_wanted
features_df["REMAINING_TARGET"] = features_df["TARGET"] + features_df["DIFFERENCE_FROM_ACCEPTANCES"]

# Adjust features_df min/max by adding a tolerance around the target number of people
# features_df["MAX"] = features_df["REMAINING_TARGET"] * ( 1 + allowed_deviation)
features_df['MAX'] = features_df.apply(
    lambda row: row['REMAINING_TARGET'] * ( 1 + allowed_deviation + additional_fow_tolerance) if row['CATEGORY'] == 'FIELD_OF_WORK' 
    else row['REMAINING_TARGET'] * ( 1 + allowed_deviation), 
    axis=1,
)
features_df['MIN'] = features_df.apply(
    lambda row: row['REMAINING_TARGET'] * ( 1 - allowed_deviation - additional_fow_tolerance) if row['CATEGORY'] == 'FIELD_OF_WORK' 
    else row['REMAINING_TARGET'] * ( 1 - allowed_deviation),
    axis=1,
)

features_df["MIN"] = features_df["MIN"].round(0).astype(int)
features_df["MAX"] = features_df["MAX"].round(0).astype(int)

# MAX: floor of 0
features_df["MAX"] = features_df["MAX"].clip(lower=0)

# MIN: 0 if MAX is 0, floor of 0, otherwise MIN should not exceed CANDIDATE_COUNT
features_df["MIN"] = features_df.apply(
    lambda row: 0 if row["MAX"] == 0
    else max(0, min(row["CANDIDATE_COUNT"], row["MIN"])),
    axis=1,
)

# rigidly adjust the maximum for some groups
features_df['MAX'] = features_df.apply(
    lambda row: row['MAX'] + max_adjustment if row['NAME'] in groups_to_adjust_max
    else row['MAX'], 
    axis=1,
)

# make the minimum equal to remaining unselected candidates to select all candidates in high need groups
features_df['MIN'] = features_df.apply(
    lambda row: row["CANDIDATE_COUNT"] if row['NAME'] in groups_high_need
    else row['MIN'], 
    axis=1,
)

# drop helper columns
features_df = features_df.drop(columns=["ACCEPTED_COUNT", "CANDIDATE_COUNT", "PROPORTIONAL_ACCEPTED_COUNT", "DIFFERENCE_FROM_ACCEPTANCES", "REMAINING_TARGET"])

features_df

## Run Sortition

Start by loading the `sortition_algorithms` library from a wheel file saved to Snowflake staging. Then prepare the sortition inputs specifically for the algorithm and run the sortition. The execution may take some time to complete.

In [ ]:
import os
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE ROLE TRANSFORM_ENGCA_PRD_READWRITECONTROL").collect()

# List all .whl files in the stage
files = session.sql("LIST @TRANSFORM_ENGCA_PRD.UTILITIES.PYTHON_WHEELS PATTERN='.*\\.whl'").collect()

# Download each file individually
os.makedirs('/tmp/wheels', exist_ok=True)
for f in files:
    filename = f['name'].split('/')[-1]
    session.file.get(f"@TRANSFORM_ENGCA_PRD.UTILITIES.PYTHON_WHEELS/{filename}", '/tmp/wheels')

# Install all downloaded wheels
!pip install /tmp/wheels/*.whl --quiet

from sortition_algorithms import (
    run_stratification,
    read_in_features,
    read_in_people,
    Settings,
)

In [ ]:
def prepare_sortition_inputs(features_df, people_df, settings, number_people_wanted):
    """Prepare inputs for the sortition algorithm."""
    features_input = features_df.copy()
    features_input.columns = features_input.columns.str.lower()

    features = read_in_features(
        list(features_input.columns),
        features_input.fillna("").to_dict(orient="records"),
        number_people_wanted,
    )[0]

    people = read_in_people(
        list(people_df.columns),
        people_df.fillna("").to_dict(orient="records"),
        features,
        settings,
    )[0]

    return features, people

In [ ]:
settings = Settings(
    id_column=id_column,
    columns_to_keep=columns_to_keep,
    selection_algorithm=selection_algorithm,
)

features, people = prepare_sortition_inputs(
    features_df, people_df, settings, number_people_wanted
)

print(f"Features loaded: {len(features)} categories")
print(f"Candidates: {len(people)}")
print(f"Number to select: {number_people_wanted}")

In [ ]:
success, selected_panels, report = run_stratification(
    features=features,
    people=people,
    number_people_wanted=number_people_wanted,
    settings=settings,
)

print(report.as_text())

if not success:
    if report.last_error():
        print(f"\nError: {report.last_error()}")
    raise RuntimeError("Sortition selection failed. See report above.")

selected_people = selected_panels[0]
print(f"\nSuccessfully selected {len(selected_people)} people")

## Output Selected Panel

Build the final selection dataframe and write it to Snowflake.

In [ ]:
selected_panel_df = (
    people_df
    .loc[people_df[id_column].isin(selected_people), [id_column]]
    .reset_index(drop=True)
)
selected_panel_df["SELECTION_TIMESTAMP"] = pd.Timestamp.now("UTC")

print(f"Selected panel shape: {selected_panel_df.shape}")
selected_panel_df.head()

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_database(TARGET_DATABASE)
session.use_schema(TARGET_SCHEMA)
snowpark_df = session.create_dataframe(selected_panel_df)

snowpark_df.write.mode("append").save_as_table(
    f"{TARGET_DATABASE}.{TARGET_SCHEMA}.{TARGET_TABLE}",
    column_order="name",
)

print(f"Successfully wrote {len(selected_panel_df)} rows to {TARGET_DATABASE}.{TARGET_SCHEMA}.{TARGET_TABLE}")